In [1]:
import io
import subprocess
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio

print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ds = ds.cast_column("audio", Audio(decode=False))

SR = 16000
pause = np.zeros(int(0.5 * SR))
waves, refs = [], []

for i in range(12):
    row = ds[i]
    w, r = sf.read(io.BytesIO(row["audio"]["bytes"]))
    assert r == SR, f"clip {i} has sample rate {r}"
    waves += [w, pause]
    refs.append(row["text"])

full = np.concatenate(waves).astype("float32")
sf.write("test_audio.wav", full, SR)
print("total duration (seconds):", round(len(full) / SR, 1))

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "lavfi", "-i", "color=c=black:s=640x360:r=10",
    "-i", "test_audio.wav",
    "-shortest", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac",
    "test_video.mp4",
], check=True)

import os
print("test_video.mp4 size (KB):", round(os.path.getsize("test_video.mp4") / 1024))

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

total duration (seconds): 136.5
test_video.mp4 size (KB): 1251


In [2]:
import subprocess
import soundfile as sf

def extract_audio(video_path, wav_path, sr=16000):
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", video_path,
        "-vn", "-ac", "1", "-ar", str(sr),
        wav_path,
    ], check=True)

extract_audio("test_video.mp4", "extracted.wav")

audio_full, rate = sf.read("extracted.wav", dtype="float32")
print("sample rate:", rate)
print("channels:", audio_full.ndim)
print("duration (seconds):", round(len(audio_full) / rate, 2))

CHUNK_SECONDS = 30

def make_chunks(audio, sr, chunk_seconds):
    size = int(chunk_seconds * sr)
    return [(start / sr, audio[start:start + size]) for start in range(0, len(audio), size)]

chunks = make_chunks(audio_full, rate, CHUNK_SECONDS)

print("number of chunks:", len(chunks))
for start, arr in chunks:
    print(f"start {start:6.1f}s | length {len(arr) / rate:5.1f}s")

sample rate: 16000
channels: 1
duration (seconds): 136.51
number of chunks: 5
start    0.0s | length  30.0s
start   30.0s | length  30.0s
start   60.0s | length  30.0s
start   90.0s | length  30.0s
start  120.0s | length  16.5s


In [3]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(name)
model = WhisperForConditionalGeneration.from_pretrained(name).to("cuda")

inputs = processor(
    [arr for _, arr in chunks],
    sampling_rate=rate,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    ids = model.generate(
        inputs.input_features,
        language="en",
        task="transcribe",
        return_timestamps=True,
    )

segments = []
for (chunk_start, arr), seq in zip(chunks, ids):
    chunk_len = len(arr) / rate
    decoded = processor.tokenizer.decode(seq, output_offsets=True)
    for seg in decoded["offsets"]:
        s, e = seg["timestamp"]
        if e is None:
            e = chunk_len
        text = seg["text"].strip()
        if text:
            segments.append({"start": chunk_start + s, "end": chunk_start + e, "text": text})

print("segments:", len(segments))
for seg in segments:
    print(f"{seg['start']:7.2f} -> {seg['end']:7.2f} | {seg['text']}")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


segments: 25
   0.00 ->    7.08 | Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.
   7.08 ->   12.28 | Nor is Mr. Quilter's manner less interesting than his matter.
  12.28 ->   17.84 | He tells us that at this festive season of the year, with Christmas and roast beef looming
  17.84 ->   25.28 | before us, similes drawn from eating and its results occur most readily to the mind.
  25.28 ->   30.04 | He has grave doubts whether Sir Frederick Layton's work is really Greek after all.
  30.00 ->   35.68 | wall, and can discover in it but little of rocky Ithaca.
  35.68 ->   41.80 | Linnell's pictures are a sort of Up Guards and Adam paintings, and Mason's exquisite
  41.80 ->   46.68 | idles are as national as a jingle poem.
  46.68 ->   52.32 | Mr. Birkett Foster's landscapes smile at one much in the same way that Mr. Carker used
  52.32 ->   54.92 | to flash his teeth.
  54.92 ->   60.02 | And Mr. John Collier gives his sitter a cheerful slap on 

In [4]:
import textwrap

def fmt_time(t):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

# 1. a cue may not end after the next one starts
segs = [dict(s) for s in sorted(segments, key=lambda x: x["start"])]
for a, b in zip(segs, segs[1:]):
    if a["end"] > b["start"]:
        a["end"] = b["start"]

# 2. split long text into cues of at most 2 lines x 42 characters
def split_text(text, n):
    words = text.split()
    parts, cur = [], []
    target = len(text) / n
    for w in words:
        cur.append(w)
        if len(" ".join(cur)) >= target and len(parts) < n - 1:
            parts.append(" ".join(cur))
            cur = []
    if cur:
        parts.append(" ".join(cur))
    return parts

def make_cues(segs, width=42, max_lines=2):
    cues = []
    for seg in segs:
        n = 1
        while True:
            parts = split_text(seg["text"], n)
            wrapped = [textwrap.wrap(p, width=width) for p in parts]
            if all(len(w) <= max_lines for w in wrapped) or n >= len(seg["text"].split()):
                break
            n += 1
        total = sum(len(p) for p in parts)
        t, dur = seg["start"], seg["end"] - seg["start"]
        for p, w in zip(parts, wrapped):
            end = t + dur * len(p) / total
            cues.append({"start": t, "end": end, "text": "\n".join(w)})
            t = end
    return cues

cues = make_cues(segs)

def to_srt(cues):
    return "\n".join(
        f"{i}\n{fmt_time(c['start'])} --> {fmt_time(c['end'])}\n{c['text']}\n"
        for i, c in enumerate(cues, start=1)
    )

srt_text = to_srt(cues)
with open("test_video.srt", "w", encoding="utf-8") as f:
    f.write(srt_text)

durations = [c["end"] - c["start"] for c in cues]
print("cues:", len(cues))
print("longest cue (s):", round(max(durations), 2))
print("shortest cue (s):", round(min(durations), 2))
print("max lines in a cue:", max(c["text"].count("\n") + 1 for c in cues))
print("overlaps left:", sum(a["end"] > b["start"] + 1e-6 for a, b in zip(cues, cues[1:])))
print()
print(srt_text[:700])

cues: 36
longest cue (s): 6.68
shortest cue (s): 0.84
max lines in a cue: 2
overlaps left: 0

1
00:00:00,000 --> 00:00:03,988
Mr. Quilter is the apostle of the middle
classes,

2
00:00:03,988 --> 00:00:07,080
and we are glad to welcome his gospel.

3
00:00:07,080 --> 00:00:12,280
Nor is Mr. Quilter's manner less
interesting than his matter.

4
00:00:12,280 --> 00:00:15,154
He tells us that at this festive season of
the

5
00:00:15,154 --> 00:00:17,840
year, with Christmas and roast beef
looming

6
00:00:17,840 --> 00:00:21,784
before us, similes drawn from eating and
its

7
00:00:21,784 --> 00:00:25,280
results occur most readily to the mind.

8
00:00:25,280 --> 00:00:30,000
He has grave doubts whether Sir Frederick
Layton's work is really Greek after all.

9
00:00:30,000 --> 00:00:3


In [5]:
old_cues = cues

STOPWORDS = {"the", "a", "an", "of", "to", "and", "its", "his", "in", "that", "with", "for", "at", "by"}

def dangling(cs):
    return sum(c["text"].split()[-1].lower() in STOPWORDS for c in cs)

def split_text(text, n):
    words = text.split()
    if n <= 1 or len(words) < n:
        return [text]
    ends, pos = [], 0
    for w in words:
        pos += len(w) + 1
        ends.append(pos - 1)
    cuts, prev = [], -1
    for k in range(1, n):
        ideal = len(text) * k / n
        best, best_score = None, None
        for i in range(prev + 1, len(words) - (n - k)):
            score = abs(ends[i] - ideal) - (10 if words[i][-1] in ",.;:?!" else 0)
            if best is None or score < best_score:
                best, best_score = i, score
        cuts.append(best)
        prev = best
    parts, start = [], 0
    for c in cuts:
        parts.append(" ".join(words[start:c + 1]))
        start = c + 1
    parts.append(" ".join(words[start:]))
    return parts

def fit_lines(p, width=42):
    return [p] if len(p) <= width else split_text(p, 2)

def make_cues(segs, width=42):
    out = []
    for seg in segs:
        n_words = len(seg["text"].split())
        n = 1
        while True:
            parts = split_text(seg["text"], n)
            lines = [fit_lines(p, width) for p in parts]
            ok = all(len(l) <= width for ls in lines for l in ls)
            if ok or n >= n_words:
                break
            n += 1
        total = sum(len(p) for p in parts)
        t, dur = seg["start"], seg["end"] - seg["start"]
        for p, ls in zip(parts, lines):
            end = t + dur * len(p) / total
            out.append({"start": t, "end": end, "text": "\n".join(ls)})
            t = end
    return out

cues = make_cues(segs)
srt_text = to_srt(cues)
with open("test_video.srt", "w", encoding="utf-8") as f:
    f.write(srt_text)

durations = [c["end"] - c["start"] for c in cues]
print("cues:", len(cues), "(before:", len(old_cues), ")")
print("dangling cues:", dangling(cues), "(before:", dangling(old_cues), ")")
print("longest cue (s):", round(max(durations), 2))
print("shortest cue (s):", round(min(durations), 2))
print("max lines in a cue:", max(c["text"].count("\n") + 1 for c in cues))
print("longest line (chars):", max(len(l) for c in cues for l in c["text"].split("\n")))
print("overlaps left:", sum(a["end"] > b["start"] + 1e-6 for a, b in zip(cues, cues[1:])))
print()
print(srt_text[:600])

cues: 36 (before: 36 )
dangling cues: 2 (before: 3 )
longest cue (s): 6.68
shortest cue (s): 0.84
max lines in a cue: 2
longest line (chars): 42
overlaps left: 0

1
00:00:00,000 --> 00:00:03,988
Mr. Quilter is the apostle
of the middle classes,

2
00:00:03,988 --> 00:00:07,080
and we are glad to welcome his gospel.

3
00:00:07,080 --> 00:00:12,280
Nor is Mr. Quilter's manner less
interesting than his matter.

4
00:00:12,280 --> 00:00:15,529
He tells us that at this
festive season of the year,

5
00:00:15,529 --> 00:00:17,840
with Christmas and roast beef looming

6
00:00:17,840 --> 00:00:21,426
before us, similes drawn from eating and

7
00:00:21,426 --> 00:00:25,280
its results occur most
readily to the mind.

8
00:00:25,280 --> 00:00:30,000
He has gra


In [6]:
!pip install -q jiwer

import re
import jiwer
import torch

def norm(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9' ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return re.sub(r"\bmister\b", "mr", s)

ref_all = norm(" ".join(refs))
print("sanity check (should be 0.0):", jiwer.wer(ref_all, ref_all))

# A. chunked pipeline
hyp_chunked = norm(" ".join(s["text"] for s in segments))
out_c = jiwer.process_words(ref_all, hyp_chunked)

# B. each original clip transcribed on its own
clips = waves[0::2]
inputs = processor(
    [c[: 30 * SR].astype("float32") for c in clips],
    sampling_rate=SR,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    ids = model.generate(inputs.input_features, language="en", task="transcribe")

hyp_clip = norm(" ".join(processor.batch_decode(ids, skip_special_tokens=True)))
out_p = jiwer.process_words(ref_all, hyp_clip)

print("reference words:", len(ref_all.split()))
for label, o in [("chunked ", out_c), ("per clip", out_p)]:
    print(f"{label} | WER {o.wer:.3f} | correct {o.hits} | substitutions {o.substitutions} | deletions {o.deletions} | insertions {o.insertions}")

sanity check (should be 0.0): 0.0
reference words: 303
chunked  | WER 0.066 | correct 285 | substitutions 14 | deletions 4 | insertions 2
per clip | WER 0.063 | correct 284 | substitutions 16 | deletions 3 | insertions 0


In [7]:
%%writefile video_to_srt.py
"""Video -> .srt subtitles: ffmpeg audio extraction, 30 s chunking, Whisper, cue building."""
import argparse
import os
import subprocess
import tempfile
import textwrap

import soundfile as sf
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

SR = 16000
CHUNK_SECONDS = 30
MIN_CHUNK_SECONDS = 1.0
STOPWORDS = {"the", "a", "an", "of", "to", "and", "its", "his", "in", "that", "with", "for", "at", "by"}


def extract_audio(video_path, wav_path, sr=SR):
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", video_path,
         "-vn", "-ac", "1", "-ar", str(sr), wav_path],
        check=True,
    )


def make_chunks(audio, sr, chunk_seconds=CHUNK_SECONDS):
    size = int(chunk_seconds * sr)
    chunks = [(start / sr, audio[start:start + size]) for start in range(0, len(audio), size)]
    return [(t, a) for t, a in chunks if len(a) >= MIN_CHUNK_SECONDS * sr]


def transcribe(chunks, model_name, language, batch_size=8):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
    segments = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        inputs = processor([a for _, a in batch], sampling_rate=SR, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = model.generate(inputs.input_features, language=language,
                                 task="transcribe", return_timestamps=True)
        for (chunk_start, arr), seq in zip(batch, ids):
            chunk_len = len(arr) / SR
            decoded = processor.tokenizer.decode(seq, output_offsets=True)
            for seg in decoded["offsets"]:
                s, e = seg["timestamp"]
                if e is None:
                    e = chunk_len
                text = seg["text"].strip()
                if text:
                    segments.append({"start": chunk_start + s, "end": chunk_start + e, "text": text})
    return segments


def fix_overlaps(segments):
    segs = [dict(s) for s in sorted(segments, key=lambda x: x["start"])]
    for a, b in zip(segs, segs[1:]):
        if a["end"] > b["start"]:
            a["end"] = b["start"]
    return segs


def split_text(text, n):
    words = text.split()
    if n <= 1 or len(words) < n:
        return [text]
    ends, pos = [], 0
    for w in words:
        pos += len(w) + 1
        ends.append(pos - 1)
    cuts, prev = [], -1
    for k in range(1, n):
        ideal = len(text) * k / n
        best, best_score = None, None
        for i in range(prev + 1, len(words) - (n - k)):
            score = abs(ends[i] - ideal) - (10 if words[i][-1] in ",.;:?!" else 0)
            if best is None or score < best_score:
                best, best_score = i, score
        cuts.append(best)
        prev = best
    parts, start = [], 0
    for c in cuts:
        parts.append(" ".join(words[start:c + 1]))
        start = c + 1
    parts.append(" ".join(words[start:]))
    return parts


def fit_lines(p, width=42):
    return [p] if len(p) <= width else split_text(p, 2)


def make_cues(segs, width=42):
    out = []
    for seg in segs:
        n_words = len(seg["text"].split())
        n = 1
        while True:
            parts = split_text(seg["text"], n)
            lines = [fit_lines(p, width) for p in parts]
            ok = all(len(l) <= width for ls in lines for l in ls)
            if ok or n >= n_words:
                break
            n += 1
        total = sum(len(p) for p in parts)
        t, dur = seg["start"], seg["end"] - seg["start"]
        for p, ls in zip(parts, lines):
            end = t + dur * len(p) / total
            out.append({"start": t, "end": end, "text": "\n".join(ls)})
            t = end
    return out


def fmt_time(t):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def to_srt(cues):
    return "\n".join(
        f"{i}\n{fmt_time(c['start'])} --> {fmt_time(c['end'])}\n{c['text']}\n"
        for i, c in enumerate(cues, start=1)
    )


def main():
    ap = argparse.ArgumentParser(description="Make an .srt subtitle file from a video.")
    ap.add_argument("video")
    ap.add_argument("-o", "--output", default=None)
    ap.add_argument("--model", default="openai/whisper-small")
    ap.add_argument("--language", default="en", help="Whisper language code, e.g. en, ta, hi")
    args = ap.parse_args()

    out_path = args.output or os.path.splitext(args.video)[0] + ".srt"
    with tempfile.TemporaryDirectory() as d:
        wav = os.path.join(d, "audio.wav")
        extract_audio(args.video, wav)
        audio, rate = sf.read(wav, dtype="float32")

    chunks = make_chunks(audio, rate)
    segments = transcribe(chunks, args.model, args.language)
    cues = make_cues(fix_overlaps(segments))

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(to_srt(cues))

    dangling = sum(c["text"].split()[-1].lower() in STOPWORDS for c in cues)
    print(f"chunks: {len(chunks)} | segments: {len(segments)} | cues: {len(cues)} | dangling: {dangling}")
    print("wrote", out_path)


if __name__ == "__main__":
    main()

Writing video_to_srt.py


In [8]:
!python video_to_srt.py test_video.mp4 -o pipeline_test.srt
!diff test_video.srt pipeline_test.srt && echo "IDENTICAL to the notebook result"

Loading weights: 100%|██████████████████████| 479/479 [00:00<00:00, 2772.34it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_thoug